### Using full_v2 (feature exclusion by dropping of duplicate data from features)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../dataset/full_v2.csv')
df.head()

In [ ]:
predictor_list = df.columns.difference(['Activity', 'Subject'])  # Exclude target and group columns
X = df[predictor_list]  # Predictors (69 components)
y = df["Activity"]                        # Target variable
groups = df["subject"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

# Get indices for train and test sets based on Subject groups
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

In [ ]:
# Initialize and train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate on unseen subjects
y_pred = model.predict(X_test)

print("--- Classification Report (Unseen Subjects) ---")
print(classification_report(y_test, y_pred))

In [ ]:
# Compute normalized matrix (percentages along rows)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

# 2. Get unique activity labels in alphabetical order (matching confusion_matrix default)
labels = sorted(y_test.unique())

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.1%',              # Format as percentage (e.g., 70.2%)
    cmap='Blues', 
    xticklabels=labels, 
    yticklabels=labels
)

plt.title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label (Ground Truth)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()